# TechMind - Treinamento do Modelo de ML

Treina um classificador **Logistic Regression + TF-IDF** para categorizar conteúdos técnicos em 8 categorias.

**Dataset:** `ml-service/data/train.csv` — 80 exemplos sintéticos (10 por categoria) em português.

**Saída:**
- `ml-service/model.joblib` — pipeline completa (vectorizer + classificador)
- `ml-service/model_metadata.json` — versão, data de treino, categorias e threshold

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import nltk
import re
import json
from datetime import datetime
import joblib
import os

nltk.download('stopwords', quiet=True)
stopwords_pt = nltk.corpus.stopwords.words('portuguese')
print(f'{len(stopwords_pt)} stopwords carregadas')

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'train.csv')
MODEL_PATH = os.path.join('..', 'model.joblib')
METADATA_PATH = os.path.join('..', 'model_metadata.json')

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {len(df)} exemplos')
print(f'Categorias: {df["categoria"].value_counts().to_dict()}')

In [ ]:
def preprocess(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = re.sub(r'\d+', '', texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_pt and len(t) > 2]
    return ' '.join(tokens)

df['texto_limpo'] = df['texto'].apply(preprocess)
print('Pré-processamento concluído')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['texto_limpo'], df['categoria'],
    test_size=0.2, random_state=42, stratify=df['categoria']
)
print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'))
])

pipeline.fit(X_train, y_train)
print('Modelo treinado')

In [ ]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Acurácia no holdout: {acc:.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
scores = cross_val_score(pipeline, df['texto_limpo'], df['categoria'], cv=5)
print(f'Cross-validation (5-folds): {scores}')
print(f'Média: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
def extract_keywords(texto_limpo, vectorizer, top_n=5):
    """Extrai os top_n termos com maior peso TF-IDF no documento."""
    tfidf_matrix = vectorizer.transform([texto_limpo])
    feature_names = vectorizer.get_feature_names_out()
    sorted_idx = tfidf_matrix[0].toarray().argsort()[0, ::-1]
    return [feature_names[i] for i in sorted_idx[:top_n] if tfidf_matrix[0, i] > 0]

def predict_with_keywords(texto, threshold=0.5):
    texto_limpo = preprocess(texto)
    probs = pipeline.predict_proba([texto_limpo])[0]
    max_prob = probs.max()
    if max_prob < threshold:
        return 'Desconhecida', float(max_prob), []
    classe = pipeline.classes_[probs.argmax()]
    keywords = extract_keywords(texto_limpo, pipeline.named_steps['tfidf'])
    return classe, float(max_prob), keywords

testes = [
    'criar API REST com Ruby on Rails e PostgreSQL',
    'componente React com animação CSS e validação de formulário',
    'infraestrutura na AWS com Terraform e Docker',
    'algoritmo de machine learning com scikit-learn para classificação',
    'texto aleatório sobre culinária e receitas'
]

for t in testes:
    cat, prob, kws = predict_with_keywords(t)
    print(f'{cat} ({prob:.4f}): {t}')
    print(f'  keywords: {kws}')

In [ ]:
MODEL_VERSION = os.environ.get('MODEL_VERSION', 'v1')
ML_THRESHOLD = float(os.environ.get('ML_THRESHOLD', '0.5'))

metadata = {
    'version': MODEL_VERSION,
    'trained_at': datetime.now().isoformat(),
    'categories': list(pipeline.classes_),
    'threshold': ML_THRESHOLD,
    'accuracy_holdout': round(acc, 4),
}

with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

joblib.dump(pipeline, MODEL_PATH)
print(f'Modelo salvo em: {MODEL_PATH}')
print(f'Metadata salva em: {METADATA_PATH}')
print(f'Versão: {MODEL_VERSION}')
print(f'Categorias: {metadata["categories"]}')